# 基于强化学习与生成模型的室外岛屿地图自动生成 - Demo版

这是一个快速演示版本，展示完整的实验流程。

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import torch
import sys
import os

# 添加当前目录到路径
sys.path.append(os.getcwd())

# 导入自定义模块
from pcg_generator import PCGIslandGenerator
from structure_evaluator import StructureEvaluator
from rl_environment import IslandGenerationEnv

print("✅ 所有模块导入成功！")

## 步骤1: PCG基座模块演示

In [ ]:
# 初始化PCG生成器
generator = PCGIslandGenerator(map_size=64)

# 定义参数
params = {
    'f': 10,
    'A': 1.0,
    'N_octaves': 4,
    'persistence': 0.5,
    'lacunarity': 2.0,
    'seed': 42,
    'warp_strength': 0.5,
    'warp_frequency': 2,
    'falloff_radius': 32,
    'falloff_exponent': 2
}

# 生成高度图
heightmap = generator.generate_heightmap(params)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 高度图
im1 = axes[0].imshow(heightmap, cmap='terrain')
axes[0].set_title('Heightmap')
plt.colorbar(im1, ax=axes[0])

# 3D视图
ax = fig.add_subplot(133, projection='3d')
x = np.arange(0, heightmap.shape[0], 2)
y = np.arange(0, heightmap.shape[1], 2)
X, Y = np.meshgrid(x, y)
Z = heightmap[::2, ::2]
ax.plot_surface(X, Y, Z, cmap='terrain')
ax.set_title('3D View')

# 二值化（陆地/水域）
binary = (heightmap > 0.3).astype(int)
axes[1].imshow(binary, cmap='binary')
axes[1].set_title('Land/Water Binary')

plt.tight_layout()
plt.savefig('demo_pcg.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 高度图生成成功！形状: {heightmap.shape}, 范围: [{heightmap.min():.3f}, {heightmap.max():.3f}]")

## 步骤2: 结构评估模块演示

In [ ]:
# 初始化评估器
evaluator = StructureEvaluator(map_size=64)

# 评估生成的地图
metrics = evaluator.evaluate(heightmap)

print("📊 结构评估指标:")
print("=" * 50)
for key, value in metrics.items():
    print(f"{key:25s}: {value:.4f}")

# 获取特征向量
feature_vec = evaluator.get_feature_vector(heightmap)
print(f"\n特征向量: {feature_vec}")
print(f"✅ 结构评估完成！")

## 步骤3: 多个随机岛屿生成

In [ ]:
# 生成多个不同参数的岛屿
n_islands = 9
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.ravel()

for i in range(n_islands):
    # 随机参数
    random_params = {
        'f': np.random.uniform(5, 20),
        'A': np.random.uniform(0.5, 1.5),
        'N_octaves': np.random.randint(3, 6),
        'persistence': np.random.uniform(0.3, 0.7),
        'lacunarity': np.random.uniform(1.5, 2.5),
        'seed': i * 100,
        'warp_strength': np.random.uniform(0.2, 0.8),
        'warp_frequency': np.random.uniform(1, 5),
        'falloff_radius': np.random.uniform(20, 40),
        'falloff_exponent': np.random.uniform(1.5, 3)
    }
    
    island = generator.generate_heightmap(random_params)
    axes[i].imshow(island, cmap='terrain')
    axes[i].set_title(f'Island {i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('demo_multiple_islands.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ 生成了 {n_islands} 个不同的岛屿！")

## 步骤4: 强化学习环境测试

In [ ]:
# 创建环境
env = IslandGenerationEnv(map_size=64, max_steps=20)

# 重置环境
state, info = env.reset(seed=42)
print(f"初始状态: {state}")
print(f"初始奖励: {env._calculate_reward(env.current_metrics):.4f}\n")

# 运行几个随机步骤
rewards = []
for step in range(10):
    action = env.action_space.sample()  # 随机动作
    state, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    
    print(f"Step {step+1:2d}: Reward = {reward:.4f}, "
          f"Connectivity = {info['metrics']['connectivity']:.2f}, "
          f"Navigable = {info['metrics']['navigable_ratio']:.2f}")
    
    if terminated or truncated:
        break

print(f"\n平均奖励: {np.mean(rewards):.4f}")
print(f"✅ RL环境测试完成！")

## 步骤5: 简单RL训练演示（无VAE简化版）

In [ ]:
import random
from collections import deque

# 简单的Q-Learning代理（用于演示）
class SimpleAgent:
    def __init__(self, action_dim, learning_rate=0.1, discount_factor=0.95, epsilon=0.3):
        self.action_dim = action_dim
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon
        self.q_table = {}
    
    def discretize_state(self, state):
        """将连续状态离散化"""
        return tuple(np.round(state, 2))
    
    def get_action(self, state):
        """ε-greedy策略选择动作"""
        if random.random() < self.epsilon:
            return np.random.uniform(-0.1, 0.1, self.action_dim)
        else:
            state_key = self.discretize_state(state)
            if state_key not in self.q_table:
                return np.random.uniform(-0.1, 0.1, self.action_dim)
            best_action_idx = np.argmax(self.q_table[state_key])
            action = np.zeros(self.action_dim)
            action[best_action_idx] = 0.1 if best_action_idx < self.action_dim // 2 else -0.1
            return action
    
    def update(self, state, action, reward, next_state, done):
        """更新Q值"""
        state_key = self.discretize_state(state)
        next_state_key = self.discretize_state(next_state)
        
        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.action_dim)
        if next_state_key not in self.q_table:
            self.q_table[next_state_key] = np.zeros(self.action_dim)
        
        best_next_action = np.max(self.q_table[next_state_key])
        td_target = reward + (0 if done else self.gamma * best_next_action)
        td_error = td_target - self.q_table[state_key][0]
        self.q_table[state_key][0] += self.lr * td_error

# 创建环境和代理
env = IslandGenerationEnv(map_size=64, max_steps=30)
agent = SimpleAgent(action_dim=env.action_space.shape[0])

# 训练循环（少量episodes用于演示）
n_episodes = 20
episode_rewards = []

print("开始训练...")
for episode in range(n_episodes):
    state, _ = env.reset(seed=episode)
    total_reward = 0
    
    for step in range(env.max_steps):
        action = agent.get_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        
        agent.update(state, action, reward, next_state, terminated or truncated)
        
        total_reward += reward
        state = next_state
        
        if terminated or truncated:
            break
    
    episode_rewards.append(total_reward)
    
    if (episode + 1) % 5 == 0:
        avg_reward = np.mean(episode_rewards[-5:])
        print(f"Episode {episode+1}/{n_episodes}, Avg Reward: {avg_reward:.4f}")

print(f"\n✅ 训练完成！最终平均奖励: {np.mean(episode_rewards[-5:]):.4f}")

## 步骤6: 训练结果可视化

In [ ]:
# 绘制训练曲线
plt.figure(figsize=(10, 6))
plt.plot(episode_rewards, marker='o', markersize=4)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Training Curve (Simple Q-Learning)')
plt.grid(True, alpha=0.3)
plt.savefig('demo_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# 移动平均
window_size = 5
if len(episode_rewards) >= window_size:
    moving_avg = np.convolve(episode_rewards, np.ones(window_size)/window_size, mode='valid')
    
    plt.figure(figsize=(10, 6))
    plt.plot(episode_rewards, label='Raw Reward', alpha=0.5)
    plt.plot(range(window_size-1, len(episode_rewards)), moving_avg, 
             label=f'{window_size}-Episode Moving Average', linewidth=2)
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.title('Training Curve with Moving Average')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('demo_training_curve_smooth.png', dpi=150, bbox_inches='tight')
    plt.show()

print("✅ 训练曲线已保存！")

## 步骤7: 最终生成的岛屿展示

In [ ]:
# 使用训练后的策略生成最终岛屿
env = IslandGenerationEnv(map_size=64, max_steps=30)
state, _ = env.reset(seed=123)

# 使用贪婪策略
for step in range(env.max_steps):
    action = agent.get_action(state)
    state, reward, terminated, truncated, info = env.step(action)
    
    if terminated or truncated:
        break

final_heightmap = info['heightmap']
final_metrics = info['metrics']

# 可视化最终结果
fig = plt.figure(figsize=(16, 5))

# 1. 高度图
ax1 = plt.subplot(131)
im1 = ax1.imshow(final_heightmap, cmap='terrain')
ax1.set_title('Final Heightmap')
plt.colorbar(im1, ax=ax1)

# 2. 3D视图
ax2 = fig.add_subplot(132, projection='3d')
x = np.arange(0, final_heightmap.shape[0], 2)
y = np.arange(0, final_heightmap.shape[1], 2)
X, Y = np.meshgrid(x, y)
Z = final_heightmap[::2, ::2]
surf = ax2.plot_surface(X, Y, Z, cmap='terrain', edgecolor='none')
ax2.set_title('3D Terrain View')
ax2.view_init(elev=30, azim=45)

# 3. 指标雷达图
ax3 = fig.add_subplot(133, projection='polar')
categories = list(final_metrics.keys())
values = list(final_metrics.values())

# 闭合雷达图
values += values[:1]
angles = [n / float(len(categories)) * 2 * np.pi for n in range(len(categories))]
angles += angles[:1]

ax3.plot(angles, values, 'o-', linewidth=2)
ax3.fill(angles, values, alpha=0.25)
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(categories, fontsize=8)
ax3.set_title('Quality Metrics')

plt.tight_layout()
plt.savefig('demo_final_result.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 最终岛屿质量指标:")
print("=" * 50)
for key, value in final_metrics.items():
    print(f"{key:25s}: {value:.4f}")

print("\n✅ Demo完成！所有结果已保存为PNG文件。")

## 总结

这个Demo展示了完整的岛屿生成流程：
1. ✅ PCG基座生成高度图
2. ✅ 结构评估模块提取特征
3. ✅ 强化学习环境封装
4. ✅ 简单RL训练
5. ✅ 结果可视化

**下一步**：完整版Notebook将包含β-VAE表征学习和SAC算法。